In [1]:
from datasets import load_dataset

squad = load_dataset("squad")
print(squad)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [2]:
print(squad["validation"].column_names)
print(squad["validation"][0])

['id', 'title', 'context', 'question', 'answers']
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented the AFC at Super Bowl 50?', 'answers': {'text': ['Denver B

In [3]:
import pandas as pd

def squad_to_df(split):
    df = pd.DataFrame(squad[split])
    df["answer_text"] = df["answers"].apply(lambda x: x["text"][0])
    return df[["context", "question", "answer_text"]]

squad_val_df = squad_to_df("validation")
squad_val_df.to_csv("squad_val_en.csv", index=False)

print(squad_val_df.shape)

(10570, 3)


In [4]:
squad_val_sample = squad_val_df.sample(n=500, random_state=42)
squad_val_sample.to_csv("squad_val_en_500.csv", index=False)

In [5]:
hotpot = load_dataset("hotpot_qa", "distractor")
print(hotpot)

README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})


In [6]:
print(hotpot["validation"].column_names)
print(hotpot["validation"][0].keys())

['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']
dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])


In [10]:
print(hotpot["validation"][0].keys())

dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])


In [13]:
import re

from collections import defaultdict

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def hotpot_gold_context_to_text(example, max_chars=2500):

    titles = example["context"]["title"]
    all_sents = example["context"]["sentences"]

    title_to_sents = {}
    for t, sents in zip(titles, all_sents):
        title_to_sents[t] = sents

    sf = example.get("supporting_facts")

    pairs = []
    if isinstance(sf, dict) and "title" in sf and "sent_id" in sf:
        pairs = list(zip(sf["title"], sf["sent_id"]))
    elif isinstance(sf, list):
        
        pairs = [(p[0], p[1]) for p in sf]
    else:
        pairs = []

    grouped = defaultdict(list)
    for t, sent_id in pairs:
        if t in title_to_sents and isinstance(sent_id, int):
            sents = title_to_sents[t]
            if 0 <= sent_id < len(sents):
                grouped[t].append(sents[sent_id])

    chunks = []
    for t, sents in grouped.items():
        body = normalize_ws(" ".join(sents))
        chunks.append(f"[{normalize_ws(t)}] {body}")

    text = "\n".join(chunks).strip()
    return text[:max_chars]

In [14]:
def hotpot_to_df_gold(hotpot_ds, split="validation", max_chars=2500):
    rows = []
    for ex in hotpot_ds[split]:
        rows.append({
            "id": ex.get("_id", ""),
            "type": ex.get("type", ""),
            "question": ex["question"],
            "context_gold": hotpot_gold_context_to_text(ex, max_chars=max_chars),
            "answer": ex["answer"],
        })
    return pd.DataFrame(rows)

hp_val_gold = hotpot_to_df_gold(hotpot, "validation", max_chars=2500)
print(hp_val_gold.shape)
hp_val_gold.head(2)

(7405, 5)


,id,type,question,context_gold,answer
0,,comparison,Were Scott Derrickson and Ed Wood of the same ...,[Scott Derrickson] Scott Derrickson (born July...,yes
1,,bridge,What government position was held by the woman...,[Kiss and Tell (1945 film)] Kiss and Tell is a...,Chief of Protocol


In [15]:
hp_val_gold.to_csv("hotpotqa_val_gold_en_full.csv", index=False)

hp_val_gold_500 = hp_val_gold.sample(n=500, random_state=42)
hp_val_gold_500.to_csv("hotpotqa_val_gold_en_500.csv", index=False)

print("Saved HotpotQA gold-context CSVs.")

Saved HotpotQA gold-context CSVs.
